# Policy Rule Training Pipeline

This notebook combines dataset generation and model training for fine-tuning Qwen2.5-3B on Rego policy rules.

**Goals:**
- Generate 2000+ training examples
- Train robust model with consistent coding style
- Use larger base model (3B parameters)

## Steps:
1. **Setup & Configuration** - Set paths and parameters
2. **Generate Dataset** - Parse Rego files and create training examples (with augmentation)
3. **Validate Dataset** - Check dataset quality and statistics
4. **Prepare Training** - Load and tokenize data
5. **Train Model** - Fine-tune with LoRA
6. **Evaluate** - Check training results


## 0. Clone Repository

First, clone the repository if it doesn't exist locally.


In [ ]:
import subprocess
import os
from pathlib import Path

# Repository configuration
REPO_URL = "https://github.com/joejstuart/policy_training.git"
REPO_NAME = "policy_training"

# Determine where to clone (current directory or parent)
WORK_DIR = Path.cwd()

# Check if we're already in the repo (look for policy directory)
if (WORK_DIR / "policy").exists():
    # Already in the repo root
    REPO_ROOT = WORK_DIR
    print(f"Already in repository at: {REPO_ROOT}")
elif WORK_DIR.name == "qwen2.5_model" and (WORK_DIR.parent / "policy").exists():
    # In qwen2.5_model subdirectory, parent is repo root
    REPO_ROOT = WORK_DIR.parent
    print(f"Using parent directory as repository root: {REPO_ROOT}")
elif WORK_DIR.name == "qwen2.5_model":
    # In qwen2.5_model but not in repo, clone to parent
    REPO_ROOT = WORK_DIR.parent / REPO_NAME
    print(f"Repository will be cloned to: {REPO_ROOT}")
else:
    # Running from elsewhere, clone here
    REPO_ROOT = WORK_DIR / REPO_NAME
    print(f"Repository will be cloned to: {REPO_ROOT}")

# Clone or update repository (only if not already in repo)
if (REPO_ROOT / "policy").exists():
    print(f"✓ Repository found at {REPO_ROOT}")
    print("Updating repository...")
    try:
        result = subprocess.run(
            ["git", "pull"],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True
        )
        print("✓ Repository updated")
    except subprocess.CalledProcessError as e:
        print(f"⚠ Warning: Could not update repository: {e}")
        print("  Continuing with existing code...")
elif REPO_ROOT.exists():
    print(f"⚠ Directory exists but doesn't look like a repository: {REPO_ROOT}")
    print("  Attempting to clone anyway...")
    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_ROOT)],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Repository cloned to {REPO_ROOT}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error cloning repository: {e}")
        print(f"  Make sure you have SSH access to {REPO_URL}")
        print(f"  Or clone manually: git clone {REPO_URL} {REPO_ROOT}")
        raise
else:
    print(f"Cloning repository from {REPO_URL}...")
    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_ROOT)],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"✓ Repository cloned to {REPO_ROOT}")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error cloning repository: {e}")
        print(f"  Make sure you have SSH access to {REPO_URL}")
        print(f"  Or clone manually: git clone {REPO_URL} {REPO_ROOT}")
        raise

# Verify repository structure
if not (REPO_ROOT / "policy").exists():
    print(f"⚠ Warning: 'policy' directory not found in {REPO_ROOT}")
    print("  Repository may not have been cloned correctly")
    raise FileNotFoundError(f"Repository structure invalid: {REPO_ROOT}")
else:
    print(f"✓ Repository structure verified")
    
# Make REPO_ROOT available globally for subsequent cells
print(f"\nRepository ready at: {REPO_ROOT}")


## 1. Setup & Configuration


In [ ]:
import json
import os
import sys
import re
import subprocess
import tempfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from collections import defaultdict
import random

# Try to import PyTorch (optional for dataset generation)
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠ PyTorch not available - dataset generation will work, but training will be skipped")

# Try to import transformers (required for training only)
try:
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        Trainer,
        TrainingArguments,
    )
    from torch.utils.data import Dataset
    TRANSFORMERS_AVAILABLE = True
    # Set tokenizers parallelism to avoid warnings
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("⚠ Transformers not available - dataset generation will work, but training will be skipped")
    # Create dummy Dataset class for type hints
    class Dataset:
        pass

# Try to import peft (optional - will fall back to full fine-tuning if not available)
try:
    from peft import LoraConfig, get_peft_model, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False
    if TRANSFORMERS_AVAILABLE:
        print("⚠ PEFT not available - will use full fine-tuning instead of LoRA")
        print("  This requires more memory but will work. Install with: pip install peft")

# Add cloned repository to Python path
# REPO_ROOT is set in the previous cell (from cloning)
sys.path.insert(0, str(REPO_ROOT))
if (REPO_ROOT / "qwen2.5_model").exists():
    sys.path.insert(0, str(REPO_ROOT / "qwen2.5_model"))

if TRANSFORMERS_AVAILABLE:
    if PEFT_AVAILABLE:
        print("✓ All imports loaded (transformers + peft available)")
    else:
        print("✓ Imports loaded (transformers available, peft not available - will use full fine-tuning)")
else:
    print("✓ Basic imports loaded (dataset generation mode)")


In [ ]:
# Configuration
# REPO_ROOT is set in the clone cell above

POLICY_RELEASE_DIR = REPO_ROOT / "policy" / "release"
POLICY_LIB_DIR = REPO_ROOT / "policy" / "lib"
RELEASE_LIB_DIR = REPO_ROOT / "policy" / "release" / "lib"

# Dataset paths (save in notebook directory, not repo)
NOTEBOOK_DIR = Path.cwd()
TRAIN_PATH = NOTEBOOK_DIR / "train.jsonl"
EVAL_PATH = NOTEBOOK_DIR / "eval.jsonl"
DATASET_SUMMARY_PATH = NOTEBOOK_DIR / "dataset_summary.json"

# Training configuration
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Upgraded from 1.5B to 3B
OUTPUT_DIR = NOTEBOOK_DIR / "qwen2.5-3b-rego-policy-lora"
MAX_SEQ_LEN = 1024  # Can handle longer sequences with 3B model
BATCH_SIZE = 1  # Start with 1, increase if memory allows
GRAD_ACCUM_STEPS = 8  # Maintain effective batch size of 8
LEARNING_RATE = 3e-5  # Slightly lower for larger model
NUM_EPOCHS = 3  # Can increase to 5 for larger dataset
LORA_R = 32  # Increased from 16 for better capacity with 3B model
LORA_ALPHA = 64  # Increased from 32 (typically 2x LORA_R)
LORA_DROPOUT = 0.05

# Dataset generation settings
TRAIN_SPLIT = 0.9  # 90% train, 10% eval
MAX_TOKENS = 2048  # Increased from 1024 to allow larger examples
REFACTOR_RATE = 1.0  # 100% of rules get refactor examples (was 0.6 = 60%)

# Directory inclusion (for dataset expansion)
INCLUDE_TASK_POLICIES = True  # Include policy/task directory
INCLUDE_PIPELINE_POLICIES = True  # Include policy/pipeline directory
INCLUDE_STEPACTION_POLICIES = True  # Include policy/stepaction directory
INCLUDE_BUILD_TASK_POLICIES = True  # Include policy/build_task directory

# Data augmentation settings (to reach 2000+ examples)
INSTRUCTION_VARIATIONS = 2  # Generate 2 instruction variations per rule
CONTEXT_VARIATIONS = 1  # Generate 1 context variation per rule
INCLUDE_TEST_FILES = False  # Set to True to extract examples from test files
SYNTHETIC_EXAMPLES = False  # Set to True to generate synthetic examples (advanced)

# Target dataset size
TARGET_EXAMPLES = 2000  # Minimum target

print(f"Repository root: {REPO_ROOT}")
print(f"Policy release dir: {POLICY_RELEASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Dataset files will be saved to: {NOTEBOOK_DIR}")


## 2. Generate Dataset

Import the dataset generation functions and run them.


In [ ]:
# Import dataset generation functions from cloned repository
# Note: Can't use "from qwen2.5_model" because module names can't contain dots
# So we add the path and import directly

import importlib.util
import sys

# Try to import from the cloned repository's qwen2.5_model directory
dataset_module_path = None

# Check if we're in the repo and can find generate_dataset.py
if (REPO_ROOT / "qwen2.5_model" / "generate_dataset.py").exists():
    dataset_module_path = REPO_ROOT / "qwen2.5_model" / "generate_dataset.py"
elif (REPO_ROOT / "generate_dataset.py").exists():
    # Fallback: maybe generate_dataset.py is in repo root
    dataset_module_path = REPO_ROOT / "generate_dataset.py"
elif Path("generate_dataset.py").exists():
    # Fallback: current directory
    dataset_module_path = Path("generate_dataset.py")

if dataset_module_path and dataset_module_path.exists():
    # Load the module dynamically
    spec = importlib.util.spec_from_file_location("generate_dataset", dataset_module_path)
    generate_dataset = importlib.util.module_from_spec(spec)
    sys.modules["generate_dataset"] = generate_dataset
    spec.loader.exec_module(generate_dataset)
    
    # Import the functions we need
    parse_rego_file = generate_dataset.parse_rego_file
    generate_implement_example = generate_dataset.generate_implement_example
    generate_refactor_example = generate_dataset.generate_refactor_example
    example_to_jsonl = generate_dataset.example_to_jsonl
    RuleExample = generate_dataset.RuleExample
    RegoFile = generate_dataset.RegoFile
    extract_used_imports = generate_dataset.extract_used_imports
    extract_used_helpers = generate_dataset.extract_used_helpers
    build_context = generate_dataset.build_context
    
    print(f"✓ Dataset generation functions imported from {dataset_module_path}")
else:
    # Last resort: try direct import (if we're already in the right directory)
    try:
        from generate_dataset import (
            parse_rego_file,
            generate_implement_example,
            generate_refactor_example,
            example_to_jsonl,
            RuleExample,
            RegoFile,
            extract_used_imports,
            extract_used_helpers,
            build_context
        )
        print("✓ Dataset generation functions imported (direct)")
    except ImportError as e:
        print(f"❌ Could not import from generate_dataset.py: {e}")
        print(f"  Make sure the repository is cloned at: {REPO_ROOT}")
        print(f"  And that qwen2.5_model/generate_dataset.py exists")
        print(f"  Or that generate_dataset.py is in the current directory")
        raise

# Helper functions to match the notebook's expected interface
def parse_rego_files(policy_dir):
    """Parse all Rego files in a directory."""
    rego_files = []
    for rego_file in policy_dir.rglob("*.rego"):
        if "_test.rego" in rego_file.name:
            continue
        parsed = parse_rego_file(rego_file)
        if parsed and parsed.rules:
            rego_files.append(parsed)
    return rego_files

def generate_training_examples(rego_file, lib_dir, release_lib_dir):
    """Generate training examples from a parsed Rego file with augmentation."""
    examples = []
    
    # Update MAX_TOKENS in the generate_dataset module if it's accessible
    if 'generate_dataset' in sys.modules:
        gen_mod = sys.modules['generate_dataset']
        if hasattr(gen_mod, 'MAX_TOKENS'):
            gen_mod.MAX_TOKENS = MAX_TOKENS
    
    for rule in rego_file.rules:
        # Generate implement example(s)
        impl_example = generate_implement_example(rego_file, rule, "")
        if impl_example:
            examples.append(impl_example)
            
            # Generate instruction variations (if enabled)
            if INSTRUCTION_VARIATIONS > 1:
                for i in range(INSTRUCTION_VARIATIONS - 1):
                    # Create variation by slightly modifying instruction
                    variation = generate_implement_example(rego_file, rule, "")
                    if variation and variation.instruction != impl_example.instruction:
                        # Only add if it's actually different
                        examples.append(variation)
        
        # Generate refactor example (configurable rate, default 100%)
        if random.random() < REFACTOR_RATE:
            refactor_example = generate_refactor_example(rego_file, rule, "")
            if refactor_example:
                examples.append(refactor_example)
    
    return examples

def split_train_eval(examples, train_split=0.9):
    """Split examples into train and eval sets."""
    random.shuffle(examples)
    
    # Separate by task type for better distribution
    implement_examples = [e for e in examples if e.task_type == "implement"]
    refactor_examples = [e for e in examples if e.task_type == "refactor"]
    
    # Split each type
    impl_split = int(len(implement_examples) * train_split)
    ref_split = int(len(refactor_examples) * train_split)
    
    train_examples = implement_examples[:impl_split] + refactor_examples[:ref_split]
    eval_examples = implement_examples[impl_split:] + refactor_examples[ref_split:]
    
    # Ensure eval has both types
    if not any(e.task_type == "implement" for e in eval_examples) and implement_examples:
        if impl_split > 0:
            train_examples.remove(implement_examples[impl_split-1])
            eval_examples.append(implement_examples[impl_split-1])
    
    if not any(e.task_type == "refactor" for e in eval_examples) and refactor_examples:
        if ref_split > 0:
            train_examples.remove(refactor_examples[ref_split-1])
            eval_examples.append(refactor_examples[ref_split-1])
    
    return train_examples, eval_examples

def write_jsonl(examples, file_path):
    """Write examples to JSONL file."""
    with open(file_path, "w", encoding="utf-8") as f:
        for example in examples:
            f.write(example_to_jsonl(example) + "\n")

print("✓ Helper wrapper functions defined")

# Update MAX_TOKENS in generate_dataset module to allow larger examples
if 'generate_dataset' in sys.modules:
    gen_mod = sys.modules['generate_dataset']
    if hasattr(gen_mod, 'MAX_TOKENS'):
        gen_mod.MAX_TOKENS = MAX_TOKENS
        print(f"  Updated MAX_TOKENS in generate_dataset to {MAX_TOKENS}")


In [ ]:
# Parse all Rego files from multiple directories
print("Parsing Rego files from multiple directories...")
rego_files = []

# Main release directory (always included)
print(f"  Parsing {POLICY_RELEASE_DIR}...")
release_files = parse_rego_files(POLICY_RELEASE_DIR)
rego_files.extend(release_files)
print(f"    Found {len(release_files)} files")

# Additional policy directories (if enabled)
if INCLUDE_TASK_POLICIES:
    task_dir = REPO_ROOT / "policy" / "task"
    if task_dir.exists():
        print(f"  Parsing {task_dir}...")
        task_files = parse_rego_files(task_dir)
        rego_files.extend(task_files)
        print(f"    Found {len(task_files)} files")

if INCLUDE_PIPELINE_POLICIES:
    pipeline_dir = REPO_ROOT / "policy" / "pipeline"
    if pipeline_dir.exists():
        print(f"  Parsing {pipeline_dir}...")
        pipeline_files = parse_rego_files(pipeline_dir)
        rego_files.extend(pipeline_files)
        print(f"    Found {len(pipeline_files)} files")

if INCLUDE_STEPACTION_POLICIES:
    stepaction_dir = REPO_ROOT / "policy" / "stepaction"
    if stepaction_dir.exists():
        print(f"  Parsing {stepaction_dir}...")
        stepaction_files = parse_rego_files(stepaction_dir)
        rego_files.extend(stepaction_files)
        print(f"    Found {len(stepaction_files)} files")

if INCLUDE_BUILD_TASK_POLICIES:
    build_task_dir = REPO_ROOT / "policy" / "build_task"
    if build_task_dir.exists():
        print(f"  Parsing {build_task_dir}...")
        build_task_files = parse_rego_files(build_task_dir)
        rego_files.extend(build_task_files)
        print(f"    Found {len(build_task_files)} files")

print(f"✓ Total: {len(rego_files)} Rego files parsed")
print(f"  Target: {TARGET_EXAMPLES} examples")

# Show some statistics
total_rules = sum(len(f.rules) for f in rego_files)
print(f"  Total rules found: {total_rules}")

# Show packages
packages = set(f.package for f in rego_files)
print(f"  Packages: {len(packages)}")
print(f"  Sample packages: {list(packages)[:5]}")


In [ ]:
# Generate training examples
print("Generating training examples...")
examples = []

for rego_file in rego_files:
    file_examples = generate_training_examples(rego_file, POLICY_LIB_DIR, RELEASE_LIB_DIR)
    examples.extend(file_examples)
    if len(examples) % 50 == 0:
        print(f"  Generated {len(examples)} examples...")

print(f"✓ Generated {len(examples)} total examples")

# Count by task type
task_types = defaultdict(int)
for ex in examples:
    task_types[ex.task_type] += 1
print(f"  Task types: {dict(task_types)}")

# Check if we've reached target
total_examples = len(examples)
print(f"\nDataset size: {total_examples} examples")
if total_examples < TARGET_EXAMPLES:
    print(f"  ⚠ Below target of {TARGET_EXAMPLES} examples")
    print(f"  Consider:")
    print(f"    - Increasing INSTRUCTION_VARIATIONS (current: {INSTRUCTION_VARIATIONS})")
    print(f"    - Enabling INCLUDE_TEST_FILES (current: {INCLUDE_TEST_FILES})")
    print(f"    - Adding more policy directories")
else:
    print(f"  ✓ Reached target of {TARGET_EXAMPLES} examples!")


In [ ]:
# Skip validation - use all examples as-is
print("Skipping validation - using all examples as generated...")
valid_examples = examples
invalid_count = 0

print(f"✓ Using {len(valid_examples)} examples (no validation performed)")


In [ ]:
# Split into train/eval
train_examples, eval_examples = split_train_eval(valid_examples, TRAIN_SPLIT)
print(f"✓ Split: {len(train_examples)} train, {len(eval_examples)} eval")

# Write to JSONL files
write_jsonl(train_examples, TRAIN_PATH)
write_jsonl(eval_examples, EVAL_PATH)
print(f"✓ Wrote {TRAIN_PATH}")
print(f"✓ Wrote {EVAL_PATH}")

# Create summary
summary = {
    "total_examples": len(valid_examples),
    "train_examples": len(train_examples),
    "eval_examples": len(eval_examples),
    "task_types": dict(task_types),
    "invalid_count": invalid_count
}
with open(DATASET_SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Wrote {DATASET_SUMMARY_PATH}")


## 3. Validate Dataset

Check dataset statistics and sample examples.


In [ ]:
# Load and display summary
with open(DATASET_SUMMARY_PATH) as f:
    summary = json.load(f)

print("Dataset Summary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Sample a few examples
print("\nSample Training Examples:\n")
with open(TRAIN_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:  # Show first 3
            break
        example = json.loads(line)
        print(f"Example {i+1} ({example['task_type']}):")
        print(f"  Instruction: {example['instruction'][:100]}...")
        print(f"  Context length: {len(example.get('context', ''))} chars")
        print(f"  Output code length: {len(example['output_code'])} chars")
        print()


## 4. Prepare Training

Load tokenizer, create dataset class, and prepare data loaders.

**Note:** This section requires `transformers` and `torch`. If they're not available, skip to the end to view your generated dataset.


In [ ]:
# Load tokenizer
if not TRANSFORMERS_AVAILABLE:
    print("⚠ Transformers not available. Skipping training preparation.")
    print("  Dataset generation is complete. Install transformers to continue with training.")
    raise ImportError("transformers is required for training. Install with: pip install transformers peft torch")

print(f"Loading tokenizer from {MODEL_NAME}...")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, 
        trust_remote_code=True,
        resume_download=True,  # Resume interrupted downloads
        force_download=False,  # Set to True if you get consistency errors
    )
except Exception as e:
    error_msg = str(e)
    print(f"\n❌ Error loading tokenizer: {error_msg}")
    if "consistency" in error_msg.lower() or "checksum" in error_msg.lower() or "file should be of size" in error_msg.lower():
        print("\n⚠ Consistency check error detected!")
        print("  The downloaded file is corrupted. Fixing by forcing re-download...")
        print("\n  Retrying with force_download=True...")
        try:
            tokenizer = AutoTokenizer.from_pretrained(
                MODEL_NAME,
                trust_remote_code=True,
                force_download=True,  # Force re-download to fix corruption
            )
            print("✓ Tokenizer downloaded successfully after retry")
        except Exception as e2:
            print(f"\n❌ Retry also failed: {e2}")
            print("\nManual fix options:")
            print("  1. Clear HuggingFace cache (see cell 4.5)")
            print("  2. Check internet connection")
            print("  3. Try again later (HuggingFace may be having issues)")
            raise
    else:
        raise

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Tokenizer loaded (vocab size: {len(tokenizer)})")


In [ ]:
# System prompt
QWEN_SYSTEM_PROMPT = (
    "You are an expert Rego/OPA policy assistant. "
    "You follow instructions carefully and emit valid Rego code using "
    "Conforma's preferred patterns (deny contains result, METADATA, result_helper, etc). "
    "Only use helpers that are provided in the context - never invent new helper functions."
)

def build_messages_from_example(example):
    """Build chat messages from policy training example."""
    messages = [
        {"role": "system", "content": QWEN_SYSTEM_PROMPT}
    ]
    
    # Build user message
    user_parts = []
    
    if "context" in example:
        user_parts.append(example["context"])
    
    if "instruction" in example:
        user_parts.append("\n" + example["instruction"])
    
    if example.get("task_type") == "refactor" and "input_code" in example:
        user_parts.append("\n\nCode to refactor:\n```rego\n" + example["input_code"] + "\n```")
    
    user_content = "\n".join(user_parts)
    messages.append({"role": "user", "content": user_content})
    
    if "output_code" in example:
        messages.append({"role": "assistant", "content": example["output_code"]})
    
    return messages


In [ ]:
# Dataset class
class PolicyDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.examples = []
        
        # Load examples
        jsonl_path = Path(jsonl_path)
        if not jsonl_path.exists():
            raise FileNotFoundError(f"Dataset file not found: {jsonl_path}")
        
        with open(jsonl_path) as f:
            for line in f:
                if line.strip():
                    self.examples.append(json.loads(line))
        
        # Pre-tokenize all examples
        print(f"Pre-tokenizing {len(self.examples)} examples...")
        self.tokenized = []
        for i, example in enumerate(self.examples):
            if i % 50 == 0:
                print(f"  Tokenized {i}/{len(self.examples)}...")
            
            messages = build_messages_from_example(example)
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding=False
            )
            
            self.tokenized.append({
                "input_ids": encoded["input_ids"],
                "attention_mask": encoded["attention_mask"]
            })
        
        print(f"✓ Pre-tokenization complete")
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        return self.tokenized[idx]

print("✓ Dataset class defined")


In [ ]:
# Create datasets
print("Creating training dataset...")
train_dataset = PolicyDataset(TRAIN_PATH, tokenizer, max_length=MAX_SEQ_LEN)

print("\nCreating eval dataset...")
eval_dataset = PolicyDataset(EVAL_PATH, tokenizer, max_length=MAX_SEQ_LEN)

print(f"\n✓ Datasets ready:")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Eval: {len(eval_dataset)} examples")


In [ ]:
# Fix HuggingFace cache issues (run if you get consistency check errors)
# The model loading cells will automatically retry with force_download=True,
# but if that doesn't work, you can manually clear the cache here:

CLEAR_CACHE = False  # Set to True to clear cache for the model

if CLEAR_CACHE:
    try:
        from huggingface_hub import scan_cache_dir
        
        # Scan cache
        cache_info = scan_cache_dir()
        print(f"Cache size: {cache_info.size_on_disk_str}")
        
        # Find and delete model files
        deleted = False
        for repo in cache_info.repos:
            if MODEL_NAME.split('/')[-1] in str(repo):
                print(f"Found model in cache: {repo}")
                # Delete all revisions for this repo
                for revision in repo.revisions:
                    print(f"  Deleting revision: {revision.revision_hash}")
                    cache_info.delete_revisions(revision.revision_hash)
                    deleted = True
        
        if deleted:
            print("✓ Cache cleared for model. Re-run model loading cell.")
        else:
            print("⚠ Model not found in cache (or already cleared)")
    except ImportError:
        print("⚠ huggingface_hub not available. Install with: pip install huggingface_hub")
    except Exception as e:
        print(f"Error clearing cache: {e}")
        print("You can also manually delete: ~/.cache/huggingface/hub/")
else:
    print("💡 To clear cache manually:")
    print("  1. Set CLEAR_CACHE = True above and run this cell")
    print("  2. Or manually delete: ~/.cache/huggingface/hub/")
    print("  3. The model loading cells will auto-retry with force_download=True")


## 5. Train Model

Load base model, configure LoRA (or full fine-tuning), and start training.

**Requirements:**
- `transformers` and `torch` are **required**
- `peft` is **optional** - if not available, will use full fine-tuning instead of LoRA

**Full fine-tuning vs LoRA:**
- **LoRA (with peft)**: Memory efficient, faster, less overfitting, small adapter files
- **Full fine-tuning (no peft)**: Uses more memory, slower, higher overfitting risk, large model files

If transformers/torch aren't available:
1. Generate the dataset (cells 0-11) without transformers
2. Copy `train.jsonl` and `eval.jsonl` to a machine with transformers
3. Run training cells (15+) on that machine


In [ ]:
# Load base model
if not TRANSFORMERS_AVAILABLE or not TORCH_AVAILABLE:
    print("⚠ Transformers/PyTorch not available. Skipping model training.")
    print("  Your dataset files are ready:")
    print(f"    - {TRAIN_PATH}")
    print(f"    - {EVAL_PATH}")
    print("  Copy these to a machine with transformers installed to train the model.")
    raise ImportError("transformers and torch are required for training")

print(f"Loading base model: {MODEL_NAME}...")

# Detect device
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Try bfloat16 first (more memory efficient), fall back to float32 if needed
# Note: We'll disable mixed precision in TrainingArguments to avoid gradient scaling issues
if device == "cuda":
    # Try bfloat16 for CUDA (uses half the memory of float32)
    # We'll handle the gradient scaling issue by disabling fp16/bf16 in TrainingArguments
    dtype = torch.bfloat16
    print("⚠ Using bfloat16 for CUDA (memory efficient)")
    print("  Mixed precision disabled in TrainingArguments to avoid gradient scaling issues")
elif device == "mps":
    dtype = torch.bfloat16
else:
    dtype = torch.float32

# Load model with better error handling for download issues
try:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map={"": device} if device != "cpu" else None,
        trust_remote_code=True,
        # Add retry and resume capabilities
        resume_download=True,  # Resume interrupted downloads
        force_download=False,  # Set to True if you get consistency errors
        local_files_only=False,  # Allow downloading if not cached
    )
except Exception as e:
    error_msg = str(e)
    print(f"\n❌ Error loading model: {error_msg}")
    
    # Check for consistency/checksum errors
    if "consistency" in error_msg.lower() or "checksum" in error_msg.lower() or "file should be of size" in error_msg.lower():
        print("\n⚠ Consistency check error detected!")
        print("  The downloaded file is corrupted. Fixing by forcing re-download...")
        print("\n  Retrying with force_download=True...")
        print("  (This will re-download all model files - may take a few minutes)")
        try:
            base_model = AutoModelForCausalLM.from_pretrained(
                MODEL_NAME,
                torch_dtype=dtype,
                device_map={"": device} if device != "cpu" else None,
                trust_remote_code=True,
                force_download=True,  # Force re-download to fix corruption
            )
            print("✓ Model downloaded successfully after retry")
        except Exception as e2:
            print(f"\n❌ Retry also failed: {e2}")
            print("\nManual fix options:")
            print("  1. Clear HuggingFace cache (see cell 4.5)")
            print("  2. Check internet connection")
            print("  3. Try again later (HuggingFace may be having issues)")
            print("  4. Use fallback model: MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'")
            raise
    
    elif "authentication" in error_msg.lower() or "gated" in error_msg.lower():
        print("\n⚠ Authentication required. This model may be gated.")
        print("  Solution: Run 'huggingface-cli login' and accept the model terms")
    
    elif "not found" in error_msg.lower():
        print(f"\n⚠ Model not found: {MODEL_NAME}")
        print("  Possible issues:")
        print("  1. Model name is incorrect")
        print("  2. Model doesn't exist on HuggingFace")
        print("  3. Model requires authentication")
        print("\n  Try verifying the model name:")
        print("  - Check: https://huggingface.co/Qwen/Qwen2.5-3B-Instruct")
        print("  - Or use: Qwen/Qwen2.5-1.5B-Instruct (fallback)")
    
    else:
        # Re-raise other errors
        raise

# Disable cache for training (required with gradient checkpointing)
if hasattr(base_model, "config"):
    base_model.config.use_cache = False

if device == "cpu":
    base_model = base_model.to(device)

# Clear CUDA cache to free up memory
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"  GPU memory after loading: {torch.cuda.memory_allocated()/1e9:.2f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")

print(f"✓ Base model loaded (dtype: {dtype})")


In [ ]:
# Configure LoRA or use full fine-tuning
if PEFT_AVAILABLE:
    # Use LoRA (memory efficient, recommended)
    print("Configuring LoRA adapters...")
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none"
    )
    
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()
    print(f"✓ LoRA configured and applied")
else:
    # Fall back to full fine-tuning (no LoRA)
    print("⚠ PEFT not available - using full fine-tuning instead of LoRA")
    print("  This will:")
    print("    - Use more memory (train all 1.5B parameters)")
    print("    - Be slower to train")
    print("    - Save the entire model (~3GB) instead of just adapters (~50MB)")
    print("    - Have higher risk of overfitting on small datasets")
    print("  Consider installing peft: pip install peft")
    print()
    
    # Use base model directly - all parameters will be trainable
    model = base_model
    
    # Count trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Model parameters:")
    print(f"  Total: {total_params/1e9:.2f}B")
    print(f"  Trainable: {trainable_params/1e9:.2f}B ({100*trainable_params/total_params:.1f}%)")
    print(f"✓ Full fine-tuning mode configured")


In [ ]:
# Training arguments
# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Adjust batch size for full fine-tuning (uses more memory)
effective_batch_size = BATCH_SIZE
if not PEFT_AVAILABLE:
    print("⚠ Full fine-tuning mode: using minimal batch size to save memory")
    effective_batch_size = 1  # Use batch size 1 for full fine-tuning
    print(f"  Batch size: {BATCH_SIZE} → {effective_batch_size}")
    print(f"  Effective batch (with grad accum): {effective_batch_size * GRAD_ACCUM_STEPS}")
    print(f"  Sequence length: {MAX_SEQ_LEN} (reduced to save memory)")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=effective_batch_size,
    per_device_eval_batch_size=effective_batch_size,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,
    logging_steps=10,
    eval_steps=200,  # Less frequent evaluation to reduce I/O
    save_steps=500,  # Less frequent saves to reduce disk I/O (was 100)
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=False,  # Disable to avoid loading issues if checkpoint save fails
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,  # Critical for memory savings
    # Disable mixed precision - model dtype is already set, don't use fp16/bf16 in TrainingArguments
    # This avoids the "not implemented for BFloat16" error with gradient scaling
    fp16=False,
    bf16=False,
    # Additional memory optimizations
    dataloader_pin_memory=False,  # Don't pin memory (saves GPU memory)
    dataloader_num_workers=0,  # No multiprocessing (saves memory)
    max_grad_norm=1.0,  # Gradient clipping
    # Reduce checkpoint size to avoid disk I/O issues
    save_only_model=True,  # Only save model, not optimizer/scheduler (saves space and I/O)
    save_total_limit=2,  # Keep only last 2 checkpoints (saves disk space)
    report_to="none",
    remove_unused_columns=False
)

print("✓ Training arguments configured")
print(f"  Save steps: {training_args.save_steps} (less frequent to reduce disk I/O)")
print(f"  Save only model: {training_args.save_only_model} (skips optimizer/scheduler to save space)")
print(f"  Max checkpoints: {training_args.save_total_limit}")
if not PEFT_AVAILABLE:
    print("  Using gradient checkpointing to reduce memory usage")

# Check disk space
import shutil
total, used, free = shutil.disk_usage(OUTPUT_DIR)
print(f"\nDisk space at {OUTPUT_DIR}:")
print(f"  Total: {total/1e9:.2f} GB")
print(f"  Used: {used/1e9:.2f} GB")
print(f"  Free: {free/1e9:.2f} GB")
if free < 5e9:  # Less than 5GB free
    print("  ⚠ Warning: Low disk space! Checkpoints may fail to save.")


In [ ]:
# Data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("✓ Data collator created")


In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("✓ Trainer created")


In [ ]:
# Start training
print("Starting training...")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Training for {NUM_EPOCHS} epochs")
print(f"Batch size: {effective_batch_size} (effective: {effective_batch_size * GRAD_ACCUM_STEPS})")
print(f"Sequence length: {MAX_SEQ_LEN}")
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e9:.2f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.2f}GB")
print()

trainer.train()

print("\n✓ Training complete!")


In [ ]:
# Save final model with error handling
print(f"Saving model to {OUTPUT_DIR}...")
try:
    trainer.save_model()
    tokenizer.save_pretrained(OUTPUT_DIR)
    
    if PEFT_AVAILABLE:
        print(f"✓ Model saved to {OUTPUT_DIR}")
        print(f"  LoRA adapters saved (small file, ~10-50MB)")
        print(f"  To load: use PeftModel.from_pretrained(base_model, '{OUTPUT_DIR}')")
    else:
        print(f"✓ Model saved to {OUTPUT_DIR}")
        print(f"  Full model saved (large file, ~3GB)")
        print(f"  To load: use AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR}')")
except Exception as e:
    print(f"⚠ Error saving final model: {e}")
    print("  Model state is still in memory. Try saving manually:")
    print(f"  trainer.save_model('{OUTPUT_DIR}/final')")
    print("  Or check disk space and permissions.")


## 6. Evaluate

Check training metrics and sample outputs.


In [ ]:
# Load training history
checkpoints = list(OUTPUT_DIR.glob("checkpoint-*"))

if checkpoints:
    try:
        latest_checkpoint = max(checkpoints, key=lambda p: int(p.name.split("-")[1]))
        trainer_state_path = latest_checkpoint / "trainer_state.json"
        
        if trainer_state_path.exists():
            with open(trainer_state_path) as f:
                state = json.load(f)
            
            print("Training History (last 10 entries):")
            if "log_history" in state:
                for entry in state["log_history"][-10:]:
                    if "loss" in entry:
                        step = entry.get("step", "?")
                        loss = entry.get("loss", "?")
                        eval_loss = entry.get("eval_loss", "?")
                        print(f"  Step {step}: loss={loss:.4f}, eval_loss={eval_loss:.4f}")
        else:
            print("No trainer_state.json found in checkpoints")
    except (ValueError, KeyError) as e:
        print(f"Could not parse checkpoint names: {e}")
        print(f"Found {len(checkpoints)} checkpoints")
else:
    print("No checkpoints found")


In [ ]:
# Test inference on a sample
print("\nTesting inference on a sample example...")

if not EVAL_PATH.exists():
    print(f"⚠ Eval file not found: {EVAL_PATH}")
    print("  Skipping inference test")
else:
    with open(EVAL_PATH) as f:
        first_line = f.readline().strip()
        if not first_line:
            print("⚠ Eval file is empty")
        else:
            sample = json.loads(first_line)
            
            messages = build_messages_from_example(sample)
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            
            inputs = tokenizer(text, return_tensors="pt").to(device)
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    temperature=0.7,
                    do_sample=True
                )
            
            generated = tokenizer.decode(outputs[0], skip_special_tokens=False)
            assistant_text = tokenizer.apply_chat_template(
                messages + [{"role": "assistant", "content": ""}],
                tokenize=False,
                add_generation_prompt=True
            )
            
            if generated.startswith(assistant_text):
                response = generated[len(assistant_text):].strip()
            else:
                # Try to extract just the assistant response
                response = generated.split("assistant\n")[-1].strip() if "assistant\n" in generated else generated
            
            print("\nSample Input:")
            print(sample.get("instruction", "")[:200])
            print("\nGenerated Output:")
            print(response[:500])
            print("\nExpected Output:")
            print(sample.get("output_code", "")[:500])
